In [15]:
# ============================================
# Logistic Regression + GridSearchCV
# ============================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd

# 1. Define Logistic Regression model
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# 2. Define parameters for GridSearchCV
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"]
}

# 3. GridSearchCV
grid_search = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

# 4. Train model
grid_search.fit(X_train, y_train)

# 5. Best model
best_log_reg = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

print("Best Cross-validation F1 Score:")
print(grid_search.best_score_)

# 6. Predict
y_pred = best_log_reg.predict(X_test)
y_prob = best_log_reg.predict_proba(X_test)[:, 1]

# 7. Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Best Parameters:
{'C': 100, 'solver': 'liblinear'}
Best Cross-validation F1 Score:
0.4864646464646464
Accuracy: 0.9791666666666666
Precision: 0.5555555555555556
Recall: 0.8333333333333334
F1 Score: 0.6666666666666666
ROC-AUC: 0.9928774928774928

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99       234
           1       0.56      0.83      0.67         6

    accuracy                           0.98       240
   macro avg       0.78      0.91      0.83       240
weighted avg       0.98      0.98      0.98       240


Confusion Matrix:
[[230   4]
 [  1   5]]


In [ ]:
# ============================================
# Remove some features and re-evaluate
# ============================================

# Process form df agian，to obtain feature names
X_original = df.drop(columns=["depression_label"])
y_original = df["depression_label"]

X_original = pd.get_dummies(X_original, drop_first=True, dtype=int)

X_train_df, X_test_df, y_train_removed, y_test_removed = train_test_split(
    X_original,
    y_original,
    test_size=0.2,
    random_state=42,
    stratify=y_original
)

# Randomly remove 30% of features
np.random.seed(42)

num_features_to_remove = int(0.3 * X_train_df.shape[1])

features_to_remove = np.random.choice(
    X_train_df.columns,
    size=num_features_to_remove,
    replace=False
)

print("Removed features:")
print(features_to_remove)

X_train_removed = X_train_df.drop(columns=features_to_remove)
X_test_removed = X_test_df.drop(columns=features_to_remove)

# Scale again
scaler_removed = StandardScaler()

X_train_removed_scaled = scaler_removed.fit_transform(X_train_removed)
X_test_removed_scaled = scaler_removed.transform(X_test_removed)

# Use the same GridSearchCV process again
grid_search_removed = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_search_removed.fit(X_train_removed_scaled, y_train_removed)

best_log_reg_removed = grid_search_removed.best_estimator_

print("Best Parameters After Feature Removal:")
print(grid_search_removed.best_params_)

print("Best Cross-validation F1 Score After Feature Removal:")
print(grid_search_removed.best_score_)

# Predict again
y_pred_removed = best_log_reg_removed.predict(X_test_removed_scaled)
y_prob_removed = best_log_reg_removed.predict_proba(X_test_removed_scaled)[:, 1]

# Evaluate again
print("Accuracy After Feature Removal:", accuracy_score(y_test_removed, y_pred_removed))
print("Precision After Feature Removal:", precision_score(y_test_removed, y_pred_removed, zero_division=0))
print("Recall After Feature Removal:", recall_score(y_test_removed, y_pred_removed, zero_division=0))
print("F1 Score After Feature Removal:", f1_score(y_test_removed, y_pred_removed, zero_division=0))
print("ROC-AUC After Feature Removal:", roc_auc_score(y_test_removed, y_prob_removed))

print("\nClassification Report After Feature Removal:")
print(classification_report(y_test_removed, y_pred_removed, zero_division=0))

print("\nConfusion Matrix After Feature Removal:")
print(confusion_matrix(y_test_removed, y_pred_removed))

Removed features:
['gender_male' 'platform_usage_TikTok' 'age'
 'social_interaction_level_low']
Best Parameters After Feature Removal:
{'C': 100, 'solver': 'liblinear'}
Best Cross-validation F1 Score After Feature Removal:
0.5039962651727359
Accuracy After Feature Removal: 0.9708333333333333
Precision After Feature Removal: 0.45454545454545453
Recall After Feature Removal: 0.8333333333333334
F1 Score After Feature Removal: 0.5882352941176471
ROC-AUC After Feature Removal: 0.9935897435897436

Classification Report After Feature Removal:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98       234
           1       0.45      0.83      0.59         6

    accuracy                           0.97       240
   macro avg       0.73      0.90      0.79       240
weighted avg       0.98      0.97      0.97       240


Confusion Matrix After Feature Removal:
[[228   6]
 [  1   5]]


In [17]:
# ============================================
# Compare before and after feature removal
# ============================================

comparison = pd.DataFrame({
    "Experiment": ["Before Feature Removal", "After Feature Removal"],
    "Best Parameters": [
        grid_search.best_params_,
        grid_search_removed.best_params_
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test_removed, y_pred_removed)
    ],
    "Precision": [
        precision_score(y_test, y_pred, zero_division=0),
        precision_score(y_test_removed, y_pred_removed, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred, zero_division=0),
        recall_score(y_test_removed, y_pred_removed, zero_division=0)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred, zero_division=0),
        f1_score(y_test_removed, y_pred_removed, zero_division=0)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test_removed, y_prob_removed)
    ]
})

display(comparison)

,Experiment,Best Parameters,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Before Feature Removal,"{'C': 100, 'solver': 'liblinear'}",0.979167,0.555556,0.833333,0.666667,0.992877
1,After Feature Removal,"{'C': 100, 'solver': 'liblinear'}",0.970833,0.454545,0.833333,0.588235,0.993590


GridSearchCV was used to tune the Logistic Regression model. The best parameters were selected based on F1 score because the dataset is imbalanced. After removing 30% of the features randomly, the model was trained and evaluated again. The comparison table shows whether the model performance improved or decreased after feature removal. If F1 score or recall decreases, it suggests that the removed features contained useful information for predicting depression_label. If the performance remains similar, it suggests that the Logistic Regression model can still make predictions using the remaining features.